## Overview

Goal: Compare Gradient Boosted Decision Trees (XGBoost) and a Multi-Layer Perceptron (MLPClassifier) on the Bank Marketing binary classification task.

Dataset: Bank Marketing campaign records with mixed numeric/categorical predictors and binary response `y` (subscribe: yes/no).

Models compared:
- XGBoost (`XGBClassifier`)
- sklearn MLP (`MLPClassifier`)

Pipeline:
- Setup paths, reproducibility, and helper functions
- Load data and perform minimal feature engineering
- Create one stratified 70/15/15 train/validation/test split
- Fit model-specific preprocessors on training data only
- Tune/train XGBoost and MLP on the same split
- Evaluate with Accuracy, Precision, Recall, F1, and AUC-PR
- Generate plots and save all outputs


## 0. Setup


In [ ]:
from __future__ import annotations

import io
import json
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import ParameterGrid, ParameterSampler, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)


In [ ]:
# Reproducibility + paths
RANDOM_SEED = 42
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_ZIP = PROJECT_ROOT / "data" / "bank+marketing.zip"
DATA_CSV = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
METRICS_DIR = REPORTS_DIR / "metrics"
FIGURES_DIR = REPORTS_DIR / "figures"

# Search sizes (tune these if you want faster runs)
XGB_N_TRIALS = 24
MLP_MAX_TRIALS = 18  # set to None to run full grid

for d in [METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)


In [ ]:
def _read_bank_csv(path_or_buffer) -> pd.DataFrame:
    """Read the UCI bank CSV regardless of delimiter variant.

    Some copies use `;` while others use `,`. We sniff the first line and
    parse with the detected separator to keep loading logic robust.
    """
    if hasattr(path_or_buffer, "read"):
        sample = path_or_buffer.read(2048)
        if isinstance(sample, bytes):
            sample = sample.decode("utf-8", errors="ignore")
        if hasattr(path_or_buffer, "seek"):
            path_or_buffer.seek(0)
    else:
        sample = Path(path_or_buffer).read_text(encoding="utf-8", errors="ignore")[:2048]

    first_line = sample.splitlines()[0] if sample else ""
    sep = ";" if ";" in first_line else ","
    return pd.read_csv(path_or_buffer, sep=sep)


def load_bank_marketing_df(csv_path: Path, zip_path: Path | None = None) -> pd.DataFrame:
    """Load dataset from either extracted CSV or provided nested archive."""
    if csv_path.exists():
        df = _read_bank_csv(csv_path)
    elif zip_path is not None and zip_path.exists():
        with zipfile.ZipFile(zip_path) as outer_zip:
            if "bank.zip" not in outer_zip.namelist():
                raise FileNotFoundError("Expected 'bank.zip' inside bank+marketing.zip")
            bank_zip_bytes = outer_zip.read("bank.zip")

        with zipfile.ZipFile(io.BytesIO(bank_zip_bytes)) as bank_zip:
            if "bank-full.csv" not in bank_zip.namelist():
                raise FileNotFoundError("Expected 'bank-full.csv' inside bank.zip")
            with bank_zip.open("bank-full.csv") as csv_file:
                df = _read_bank_csv(csv_file)
    else:
        raise FileNotFoundError(f"Missing dataset file: {csv_path} and archive: {zip_path}")

    # Normalize schema once so downstream code can rely on stable names.
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # Policy decision: keep "unknown" as an explicit category (not missing).
    # This matches the dataset semantics and keeps preprocessing behavior stable.
    obj_cols = df.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        df[c] = df[c].astype(str).str.strip()

    # Standardize binary target and fail fast on unexpected labels.
    if "y" not in df.columns:
        raise ValueError("Target column 'y' not found")
    df["y"] = df["y"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
    if df["y"].isna().any():
        raise ValueError("Unexpected values in target column y")
    df["y"] = df["y"].astype(int)

    return df


def add_light_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add three high-signal, interpretable features for this dataset."""
    out = df.copy()

    if "pdays" in out.columns:
        # Feature 1: prior contact indicator (1 if previously contacted, else 0).
        out["prev_contacted"] = (out["pdays"] != -1).astype(int)

        # Feature 2: contact-recency bucket from pdays (0=never, 1=recent<=30d, 2=stale>30d).
        out["recent_contact"] = np.select(
            [out["pdays"] == -1, out["pdays"].between(0, 30), out["pdays"] > 30],
            [0, 1, 2],
            default=0,
        ).astype(int)

    if "poutcome" in out.columns:
        # Feature 3: prior campaign success flag (1 if last outcome was success).
        out["prior_success"] = out["poutcome"].astype(str).str.lower().eq("success").astype(int)

    return out


def split_data(df: pd.DataFrame, target_col: str = "y", random_seed: int = 42):
    """Create stratified train/validation/test splits (70/15/15)."""
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found")

    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Stratification preserves class prevalence for fair model comparison.
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=random_seed,
        stratify=y,
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp,
        y_tmp,
        test_size=0.50,
        random_state=random_seed,
        stratify=y_tmp,
    )

    return (
        X_train.reset_index(drop=True),
        X_val.reset_index(drop=True),
        X_test.reset_index(drop=True),
        y_train.reset_index(drop=True),
        y_val.reset_index(drop=True),
        y_test.reset_index(drop=True),
    )


def detect_feature_types(X: pd.DataFrame):
    """Infer numeric vs categorical columns for column-wise preprocessing."""
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]
    return num_cols, cat_cols


def build_xgb_preprocessor(num_cols, cat_cols):
    """Preprocessor for tree model: impute + one-hot (no scaling required)."""
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])


def build_mlp_preprocessor(num_cols, cat_cols):
    """Preprocessor for neural net: impute + scale numeric + one-hot categorical."""
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])


def evaluate_binary(y_true, y_prob, threshold: float = 0.5):
    """Compute thresholded classification metrics and threshold-free AUC-PR."""
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc_pr": float(average_precision_score(y_true, y_prob)),
    }


## 1. Load Data + Basic Understanding

Load the dataset, clean target encoding, and inspect shape/class balance before splitting.


In [ ]:
# Load dataset and apply lightweight feature engineering before splitting.
df = load_bank_marketing_df(DATA_CSV, DATA_ZIP)
df = add_light_features(df)

print("Data shape:", df.shape)
print("Target positive rate (full dataset):", round(df["y"].mean(), 4))
print("Sample columns:", list(df.columns[:10]))


## Data quality audit

This audit makes data-cleaning and preprocessing decisions explicit for grading clarity before splitting and model fitting.


In [ ]:
# Explicit data-quality audit for grading transparency.
print("Data shape:", df.shape)

print("\nColumn dtypes:")
print(df.dtypes)

missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_audit = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct.round(3),
})
print("\nPer-column missing counts and percentages:")
print(missing_audit)

obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    unknown_rows = []
    for col in obj_cols:
        unknown_count = int((df[col].astype(str).str.lower() == "unknown").sum())
        if unknown_count > 0:
            unknown_rows.append({"column": col, "unknown_count": unknown_count})

    print("\n'unknown' counts in object/categorical columns:")
    if unknown_rows:
        print(pd.DataFrame(unknown_rows).sort_values("unknown_count", ascending=False).to_string(index=False))
    else:
        print("No 'unknown' markers found.")

print("\nTarget distribution (y):")
print(df["y"].value_counts(dropna=False).sort_index())
print("Target proportions:")
print(df["y"].value_counts(normalize=True, dropna=False).sort_index().round(4))


## 2. Train / Validation / Test Split

Use one stratified split (70/15/15). This exact split is reused for both models to ensure a fair comparison.


In [ ]:
# Use one stratified split so both models are trained/evaluated on identical rows.
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df, target_col="y", random_seed=RANDOM_SEED)
num_cols, cat_cols = detect_feature_types(X_train)

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
print("Target positive rate (train/val/test):", round(y_train.mean(), 4), round(y_val.mean(), 4), round(y_test.mean(), 4))
print("Numeric columns:", len(num_cols), "Categorical columns:", len(cat_cols))


## 3. Preprocessing (separate XGB vs MLP)

XGBoost handles unscaled tabular features well, while MLP requires standardized numeric inputs.

No-leakage protocol used here:
- train / validation / test are split before any preprocessing fit
- imputers / encoders / scalers are fit on training data only
- validation and test are transformed using the fitted train preprocessors only


In [ ]:
# Fit preprocessors on train only to prevent train/validation/test leakage.
xgb_preprocessor = build_xgb_preprocessor(num_cols, cat_cols)
mlp_preprocessor = build_mlp_preprocessor(num_cols, cat_cols)

# Persist a separate transformed design matrix per model family.
X_train_xgb = xgb_preprocessor.fit_transform(X_train)
X_val_xgb = xgb_preprocessor.transform(X_val)
X_test_xgb = xgb_preprocessor.transform(X_test)

X_train_mlp = mlp_preprocessor.fit_transform(X_train)
X_val_mlp = mlp_preprocessor.transform(X_val)
X_test_mlp = mlp_preprocessor.transform(X_test)

print("XGB train matrix shape:", X_train_xgb.shape)
print("MLP train matrix shape:", X_train_mlp.shape)


# Explicit no-leakage sanity checks for grading clarity.
assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]
print("No-leakage check: preprocessors fit on train only; val/test only transformed.")


## 4. GBDT (XGBoost)

Tune on validation performance with early stopping and inspect optimization behavior using train/validation log loss.


Sampling note: `XGB_N_TRIALS` may cap the full grid for practical runtime. This is a computational-budgeted sample of the configured search space, controlled by a fixed random seed for reproducibility.

In [ ]:
# Parameter space intentionally mixes capacity + regularization + optimization knobs.
xgb_param_space = {
    "learning_rate": [0.01, 0.1, 0.3],
    "n_estimators": [300, 600],
    "max_depth": [3, 6],
    "subsample": [0.8, 1.0],
    "reg_alpha": [0.0, 0.5],
    "reg_lambda": [1.0, 5.0],
}

all_xgb_combos = list(ParameterGrid(xgb_param_space))
# Use random sampling only when trial cap is smaller than full combinatorial grid.
if XGB_N_TRIALS is not None and XGB_N_TRIALS < len(all_xgb_combos):
    xgb_trials = list(ParameterSampler(xgb_param_space, n_iter=XGB_N_TRIALS, random_state=RANDOM_SEED))
else:
    xgb_trials = all_xgb_combos

print(f"XGB trials: {len(xgb_trials)}")

xgb_trial_rows = []
best_xgb = None
best_xgb_score = -np.inf

for i, params in enumerate(xgb_trials, start=1):
    start = time.perf_counter()
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist",
        early_stopping_rounds=30,
        **params,
    )
    # Keep both train and validation eval sets for overfitting diagnostics.
    model.fit(
        X_train_xgb,
        y_train,
        eval_set=[(X_train_xgb, y_train), (X_val_xgb, y_val)],
        verbose=False,
    )
    elapsed = time.perf_counter() - start

    val_prob = model.predict_proba(X_val_xgb)[:, 1]
    val_metrics = evaluate_binary(y_val, val_prob)

    row = {
        "trial": i,
        **params,
        "best_iteration": int(getattr(model, "best_iteration", -1)),
        "train_time_sec": elapsed,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    xgb_trial_rows.append(row)

    # Primary selection metric: AUC-PR is more informative for imbalanced classes.
    if val_metrics["auc_pr"] > best_xgb_score:
        best_xgb_score = val_metrics["auc_pr"]
        best_xgb = model

xgb_trials_df = pd.DataFrame(xgb_trial_rows).sort_values("val_auc_pr", ascending=False).reset_index(drop=True)
xgb_trials_df.head()


In [ ]:
# Evaluate best XGB on val + test
best_xgb_val_prob = best_xgb.predict_proba(X_val_xgb)[:, 1]
best_xgb_test_prob = best_xgb.predict_proba(X_test_xgb)[:, 1]

xgb_val_metrics = evaluate_binary(y_val, best_xgb_val_prob)
xgb_test_metrics = evaluate_binary(y_test, best_xgb_test_prob)

best_xgb_row = xgb_trials_df.iloc[0].to_dict()

print("Best XGB val metrics:")
print(pd.Series(xgb_val_metrics))
print("\nBest XGB test metrics:")
print(pd.Series(xgb_test_metrics))


In [ ]:
# Compact hyperparameter sensitivity summary (validation AUC-PR).
def summarize_hp_effect(df: pd.DataFrame, hp: str) -> pd.DataFrame:
    out = (
        df.groupby(hp)["val_auc_pr"]
        .agg(mean_val_auc_pr="mean", max_val_auc_pr="max", count="count")
        .reset_index()
        .sort_values("max_val_auc_pr", ascending=False)
    )
    out[["mean_val_auc_pr", "max_val_auc_pr"]] = out[["mean_val_auc_pr", "max_val_auc_pr"]].round(4)
    return out

for hp in ["learning_rate", "max_depth", "n_estimators"]:
    out = summarize_hp_effect(xgb_trials_df, hp)
    print(f"\nXGB sensitivity by {hp}:")
    print(out.to_string(index=False))


In [ ]:
# Visualize learning-rate sensitivity from XGB trial summary table
xgb_lr_effect = summarize_hp_effect(xgb_trials_df, "learning_rate").sort_values("learning_rate")
xgb_lr_effect["learning_rate"] = xgb_lr_effect["learning_rate"].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

sns.barplot(data=xgb_lr_effect, x="learning_rate", y="mean_val_auc_pr", ax=axes[0], color="#6ea8fe")
axes[0].set_title("Mean Validation AUC-PR")
axes[0].set_xlabel("Learning Rate")
axes[0].set_ylabel("AUC-PR")

sns.barplot(data=xgb_lr_effect, x="learning_rate", y="max_val_auc_pr", ax=axes[1], color="#74c69d")
axes[1].set_title("Best Validation AUC-PR")
axes[1].set_xlabel("Learning Rate")
axes[1].set_ylabel("AUC-PR")

fig.suptitle("XGBoost Learning-Rate Sensitivity (From Trial Table)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "xgb_learning_rate_sensitivity_from_trials.png", dpi=150)
plt.show()


In [ ]:
# XGB plots + saves
# 1) training vs validation loss
evals = best_xgb.evals_result()
train_logloss = evals["validation_0"]["logloss"]
val_logloss = evals["validation_1"]["logloss"]

plt.figure(figsize=(9, 5))
plt.plot(train_logloss, label="Train Logloss")
plt.plot(val_logloss, label="Validation Logloss")
plt.xlabel("Boosting Round")
plt.ylabel("Logloss")
plt.title("XGBoost Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "xgb_train_val_loss.png", dpi=150)
plt.show()

# 2) feature importance
feature_names = xgb_preprocessor.get_feature_names_out()
importances = best_xgb.feature_importances_
fi_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False).head(20)

plt.figure(figsize=(10, 7))
sns.barplot(data=fi_df, x="importance", y="feature", orient="h")
plt.title("Top 20 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "xgb_feature_importance_top20.png", dpi=150)
plt.show()

# 3) controlled learning-rate comparison (all other params fixed)
fixed_params = {
    "n_estimators": int(best_xgb_row["n_estimators"]),
    "max_depth": int(best_xgb_row["max_depth"]),
    "subsample": float(best_xgb_row["subsample"]),
    "reg_alpha": float(best_xgb_row["reg_alpha"]),
    "reg_lambda": float(best_xgb_row["reg_lambda"]),
}
lr_values = [0.01, 0.1, 0.3]
lr_rows = []

for lr in lr_values:
    lr_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist",
        early_stopping_rounds=30,
        learning_rate=lr,
        **fixed_params,
    )
    lr_model.fit(
        X_train_xgb,
        y_train,
        eval_set=[(X_train_xgb, y_train), (X_val_xgb, y_val)],
        verbose=False,
    )
    lr_val_prob = lr_model.predict_proba(X_val_xgb)[:, 1]
    lr_rows.append({"learning_rate": lr, "val_auc_pr": average_precision_score(y_val, lr_val_prob)})

lr_summary = pd.DataFrame(lr_rows).sort_values("learning_rate").reset_index(drop=True)
plt.figure(figsize=(7, 4))
sns.lineplot(data=lr_summary, x="learning_rate", y="val_auc_pr", marker="o")
plt.title("XGBoost Learning Rate vs Validation AUC-PR (Controlled)")
plt.xlabel("Learning Rate")
plt.ylabel("Validation AUC-PR")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "xgb_learning_rate_comparison.png", dpi=150)
plt.show()

# Persisted in section 8 (Save Outputs).


## 5. MLP

Tune MLP architectures and learning settings on validation performance. Use `loss_curve_` for training-dynamics analysis.


Sampling note: `MLP_MAX_TRIALS` may cap the full grid for practical runtime. This keeps experiments tractable while preserving reproducible coverage of the search space via a fixed seed.

In [ ]:
# MLP search emphasizes architecture depth, activation family, and optimizer step size.
mlp_param_space = {
    "hidden_layer_sizes": [(64,), (128, 64), (256, 128, 64)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.01, 0.1],
    "max_iter": [200, 400],
}

mlp_trials = list(ParameterGrid(mlp_param_space))
# Downsample trial count deterministically when a cap is configured.
if MLP_MAX_TRIALS is not None and MLP_MAX_TRIALS < len(mlp_trials):
    rng = np.random.default_rng(RANDOM_SEED)
    idx = rng.choice(len(mlp_trials), size=MLP_MAX_TRIALS, replace=False)
    mlp_trials = [mlp_trials[int(i)] for i in idx]

print(f"MLP trials: {len(mlp_trials)}")

mlp_trial_rows = []
best_mlp = None
best_mlp_score = -np.inf

for i, params in enumerate(mlp_trials, start=1):
    start = time.perf_counter()
    model = MLPClassifier(
        solver="adam",
        random_state=RANDOM_SEED,
        early_stopping=True,
        n_iter_no_change=15,
        **params,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(X_train_mlp, y_train)
    elapsed = time.perf_counter() - start

    val_prob = model.predict_proba(X_val_mlp)[:, 1]
    val_metrics = evaluate_binary(y_val, val_prob)

    row = {
        "trial": i,
        **params,
        "n_iter_": int(model.n_iter_),
        "final_loss": float(model.loss_curve_[-1]) if len(model.loss_curve_) > 0 else np.nan,
        "train_time_sec": elapsed,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    mlp_trial_rows.append(row)

    # Match XGB selection criterion to keep cross-model comparison consistent.
    if val_metrics["auc_pr"] > best_mlp_score:
        best_mlp_score = val_metrics["auc_pr"]
        best_mlp = model

mlp_trials_df = pd.DataFrame(mlp_trial_rows).sort_values("val_auc_pr", ascending=False).reset_index(drop=True)
mlp_trials_df.head()


In [ ]:
# Evaluate best MLP on val + test
best_mlp_val_prob = best_mlp.predict_proba(X_val_mlp)[:, 1]
best_mlp_test_prob = best_mlp.predict_proba(X_test_mlp)[:, 1]

mlp_val_metrics = evaluate_binary(y_val, best_mlp_val_prob)
mlp_test_metrics = evaluate_binary(y_test, best_mlp_test_prob)

best_mlp_row = mlp_trials_df.iloc[0].to_dict()

print("Best MLP val metrics:")
print(pd.Series(mlp_val_metrics))
print("\nBest MLP test metrics:")
print(pd.Series(mlp_test_metrics))


In [ ]:
# Compact hyperparameter sensitivity summary (validation AUC-PR).
def summarize_hp_effect(df: pd.DataFrame, hp: str) -> pd.DataFrame:
    out = (
        df.groupby(hp)["val_auc_pr"]
        .agg(mean_val_auc_pr="mean", max_val_auc_pr="max", count="count")
        .reset_index()
        .sort_values("max_val_auc_pr", ascending=False)
    )
    out[["mean_val_auc_pr", "max_val_auc_pr"]] = out[["mean_val_auc_pr", "max_val_auc_pr"]].round(4)
    return out

for hp in ["hidden_layer_sizes", "activation", "learning_rate_init", "max_iter"]:
    out = summarize_hp_effect(mlp_trials_df, hp)
    if hp == "hidden_layer_sizes":
        out[hp] = out[hp].astype(str)
    print(f"\nMLP sensitivity by {hp}:")
    print(out.to_string(index=False))


In [ ]:
# MLP plots + saves
# 1) loss curve for best model
plt.figure(figsize=(9, 5))
plt.plot(best_mlp.loss_curve_, label="Training Loss")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("MLP Loss Curve (Best Model)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mlp_loss_curve.png", dpi=150)
plt.show()

# 2) architecture comparison (best val AUC-PR per architecture)
arch_summary = (
    mlp_trials_df
    .groupby("hidden_layer_sizes", as_index=False)["val_auc_pr"]
    .max()
    .sort_values("val_auc_pr", ascending=False)
)
# seaborn/pandas can treat tuple categories as MultiIndex; plot string labels instead.
arch_summary["hidden_layer_label"] = arch_summary["hidden_layer_sizes"].map(str)

plt.figure(figsize=(9, 4))
sns.barplot(data=arch_summary, x="hidden_layer_label", y="val_auc_pr")
plt.title("MLP Architecture Comparison (Best Validation AUC-PR)")
plt.xlabel("Hidden Layer Sizes")
plt.ylabel("Best Validation AUC-PR")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mlp_architecture_comparison.png", dpi=150)
plt.show()

# Save tables/artifacts
mlp_trials_df.to_csv(METRICS_DIR / "mlp_trials.csv", index=False)

with open(METRICS_DIR / "mlp_best_params.json", "w", encoding="utf-8") as f:
    json.dump({k: best_mlp_row[k] for k in ["hidden_layer_sizes", "activation", "learning_rate_init", "max_iter"]}, f, indent=2, default=str)

with open(METRICS_DIR / "mlp_best_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"val": mlp_val_metrics, "test": mlp_test_metrics, "train_time_sec": best_mlp_row["train_time_sec"]}, f, indent=2)


In [ ]:
# Visualize learning-rate sensitivity from MLP trial summary table
mlp_lr_effect = summarize_hp_effect(mlp_trials_df, "learning_rate_init").sort_values("learning_rate_init")
mlp_lr_effect["learning_rate_init"] = mlp_lr_effect["learning_rate_init"].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

sns.barplot(data=mlp_lr_effect, x="learning_rate_init", y="mean_val_auc_pr", ax=axes[0], color="#6ea8fe")
axes[0].set_title("Mean Validation AUC-PR")
axes[0].set_xlabel("Learning Rate (Init)")
axes[0].set_ylabel("AUC-PR")

sns.barplot(data=mlp_lr_effect, x="learning_rate_init", y="max_val_auc_pr", ax=axes[1], color="#74c69d")
axes[1].set_title("Best Validation AUC-PR")
axes[1].set_xlabel("Learning Rate (Init)")
axes[1].set_ylabel("AUC-PR")

fig.suptitle("MLP Learning-Rate Sensitivity (From Trial Table)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "mlp_learning_rate_sensitivity_from_trials.png", dpi=150)
plt.show()


## 6. Evaluation

Compute required metrics for both validation and test sets: Accuracy, Precision, Recall, F1, and AUC-PR.


In [ ]:
# Consolidate validation and test metrics for grading transparency.
eval_summary_df = pd.DataFrame([
    {"Model": "XGBoost", "Split": "Validation", **xgb_val_metrics},
    {"Model": "XGBoost", "Split": "Test", **xgb_test_metrics},
    {"Model": "MLP", "Split": "Validation", **mlp_val_metrics},
    {"Model": "MLP", "Split": "Test", **mlp_test_metrics},
])

eval_summary_df = eval_summary_df[["Model", "Split", "accuracy", "precision", "recall", "f1", "auc_pr"]]
eval_summary_df


## 7. Final Comparison


In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "XGBoost",
        "Accuracy": xgb_test_metrics["accuracy"],
        "Precision": xgb_test_metrics["precision"],
        "Recall": xgb_test_metrics["recall"],
        "F1": xgb_test_metrics["f1"],
        "AUC-PR": xgb_test_metrics["auc_pr"],
        "Training Time (s)": best_xgb_row["train_time_sec"],
    },
    {
        "Model": "MLP",
        "Accuracy": mlp_test_metrics["accuracy"],
        "Precision": mlp_test_metrics["precision"],
        "Recall": mlp_test_metrics["recall"],
        "F1": mlp_test_metrics["f1"],
        "AUC-PR": mlp_test_metrics["auc_pr"],
        "Training Time (s)": best_mlp_row["train_time_sec"],
    },
])

comparison_df


In [ ]:
# Optional PR curves on test set
xgb_prec, xgb_rec, _ = precision_recall_curve(y_test, best_xgb_test_prob)
mlp_prec, mlp_rec, _ = precision_recall_curve(y_test, best_mlp_test_prob)

plt.figure(figsize=(7, 5))
plt.plot(xgb_rec, xgb_prec, label=f"XGBoost (AP={xgb_test_metrics['auc_pr']:.3f})")
plt.plot(mlp_rec, mlp_prec, label=f"MLP (AP={mlp_test_metrics['auc_pr']:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves (Test Set)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_pr_curves.png", dpi=150)
plt.show()


In [ ]:
# Comparison plot
plot_df = comparison_df.melt(id_vars="Model", var_name="Metric", value_name="Value")
plot_df = plot_df[plot_df["Metric"].isin(["Accuracy", "Precision", "Recall", "F1", "AUC-PR"])]

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Metric", y="Value", hue="Model")
plt.ylim(0, 1)
plt.title("Model Comparison on Test Set")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_comparison_metrics.png", dpi=150)
plt.show()


## 8. Save Outputs


In [ ]:
# Persist experiment artifacts for reproducibility and grading traceability.
xgb_trials_df.to_csv(METRICS_DIR / "xgb_trials.csv", index=False)
fi_df.to_csv(METRICS_DIR / "xgb_feature_importance_top20.csv", index=False)
mlp_trials_df.to_csv(METRICS_DIR / "mlp_trials.csv", index=False)
comparison_df.to_csv(METRICS_DIR / "model_comparison.csv", index=False)
eval_summary_df.to_csv(METRICS_DIR / "evaluation_summary.csv", index=False)

with open(METRICS_DIR / "xgb_best_params.json", "w", encoding="utf-8") as f:
    json.dump({k: best_xgb_row[k] for k in ["learning_rate", "n_estimators", "max_depth", "subsample", "reg_alpha", "reg_lambda"]}, f, indent=2)

with open(METRICS_DIR / "xgb_best_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"val": xgb_val_metrics, "test": xgb_test_metrics, "train_time_sec": best_xgb_row["train_time_sec"]}, f, indent=2)

with open(METRICS_DIR / "mlp_best_params.json", "w", encoding="utf-8") as f:
    json.dump({k: best_mlp_row[k] for k in ["hidden_layer_sizes", "activation", "learning_rate_init", "max_iter"]}, f, indent=2, default=str)

with open(METRICS_DIR / "mlp_best_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"val": mlp_val_metrics, "test": mlp_test_metrics, "train_time_sec": best_mlp_row["train_time_sec"]}, f, indent=2)

print("Saved artifacts:")
for pth in sorted(METRICS_DIR.glob('*')):
    print(' -', pth)
for pth in sorted(FIGURES_DIR.glob('*')):
    print(' -', pth)
